# Training-budget sweep: does the step-count effect depend on how long we train?

**How to run:** upload *only this file* to Google Colab, then Runtime -> Change runtime type -> **GPU**, then Runtime -> **Run all**. All code is embedded; no other files are needed.

For each number of diffusion steps T and each random seed, one model is trained up to the largest budget, and FID is recorded at several **checkpoints** (default 10, 30, 60 epochs). This shows whether a reliable dependence of FID on T emerges as the training budget grows.

**Runtime (default 3 budgets checkpoints x 5 T x 3 seeds, training once to 60 epochs each):** roughly 3.5-5 h on a free Colab T4. If that is too long, in the Config cell reduce SEEDS to [0, 1], or STEP_COUNTS to [100, 400, 1000], or lower CHECKPOINTS (e.g. [10, 40]).

Outputs: a FID mean +/- std table per budget, a combined budget x T table, and `budget_sweep.json`.


In [ ]:
!pip -q install diffusers umap-learn


### Embedded core (model, matched-noise schedule, DDPM sampler, Inception-FID, seeds)


In [ ]:
"""
Controlled empirical study of diffusion-process length and inference algorithm
on MNIST image synthesis.

This module contains the exact experimental code used in the paper
"Diffusion Process Length and Inference Algorithm in Denoising-Diffusion Image
Synthesis: A Controlled Empirical Study" (I. A. Kistin, Sh. M. Isaev), refactored
from the authors' original notebook into a reproducible, seed-controlled form.

The only substantive additions relative to the original study are:
  * explicit random-seed control (set_seed),
  * a multi-seed loop for uncertainty estimates on FID,
  * FID for both DDPM and DDIM across restoration-step counts,
  * a proper Inception-based FID for the conditional model
    (the original notebook used a random-projection proxy).
No modelling choice, hyper-parameter, or schedule has been changed.
"""
import math, os, random, argparse, json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.models as tv_models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# --------------------------------------------------------------------------- #
# Reproducibility
# --------------------------------------------------------------------------- #
def set_seed(seed: int):
    """Fix all RNGs for reproducible training/evaluation."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def mnist_loader(batch_size=128, train=True, to_pm1=True):
    tfm = [transforms.ToTensor()]
    if to_pm1:
        tfm.append(transforms.Lambda(lambda x: x * 2. - 1.))
    ds = datasets.MNIST(root='./data', train=train, download=True,
                        transform=transforms.Compose(tfm))
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=train,
                          num_workers=2, pin_memory=True, drop_last=False)


# --------------------------------------------------------------------------- #
# Unconditional model: custom SimpleUNet (matched-noise schedule)
# --------------------------------------------------------------------------- #
def sinusoidal_embedding(timesteps, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=timesteps.device) / half)
    args = timesteps.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb


class SimpleUNet(nn.Module):
    """Compact convolutional U-Net with additive skip connections.
    Time index -> sinusoidal embedding (dim 128) -> 2-layer MLP (SiLU),
    added at the bottleneck. ~ encoder 1->32->64->128, symmetric decoder."""
    def __init__(self, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim), nn.SiLU(),
                                      nn.Linear(time_dim, time_dim))
        self.conv0 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv1 = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(128, 128, 3, padding=1)
        self.deconv1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.out_conv = nn.Conv2d(32, 1, 3, padding=1)
        self.act = nn.SiLU()
        self.time_dim = time_dim

    def forward(self, x, t):
        t = self.time_mlp(sinusoidal_embedding(t, self.time_dim))[:, :, None, None]
        h0 = self.act(self.conv0(x))
        h1 = self.act(self.conv1(h0))
        h2 = self.act(self.conv2(h1))
        h3 = self.act(self.conv3(h2 + t))
        u1 = self.act(self.deconv1(h3)) + h1
        u2 = self.act(self.deconv2(u1)) + h0
        return self.out_conv(u2)


def make_schedule(T, beta_1=1e-4, target_alpha_bar_T=1e-3):
    """Linear beta schedule from beta_1 to beta_T, where beta_T is found by
    binary search so that the terminal cumulative signal retention
    alpha_bar_T is (approximately) equal to target_alpha_bar_T for every T.
    This is the 'matched total noise' control central to the study."""
    low, high = beta_1, 0.999
    for _ in range(100):
        mid = (low + high) / 2
        betas = torch.linspace(beta_1, mid, T)
        alpha_bar = torch.cumprod(1. - betas, dim=0)
        if alpha_bar[-1] > target_alpha_bar_T:
            low = mid
        else:
            high = mid
    betas = torch.linspace(beta_1, high, T, device=device)
    alphas = 1. - betas
    alpha_bar = torch.cumprod(alphas, dim=0)
    return {'T': T, 'betas': betas, 'alphas': alphas, 'alpha_bar': alpha_bar,
            'sqrt_alpha_bar': torch.sqrt(alpha_bar),
            'sqrt_one_minus_alpha_bar': torch.sqrt(1. - alpha_bar)}


def train_ddpm(T, epochs, loader, lr=2e-4):
    schedule = make_schedule(T)
    model = SimpleUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    for _ in range(epochs):
        for x, _ in loader:
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_t = (schedule['sqrt_alpha_bar'][t].view(-1, 1, 1, 1) * x +
                   schedule['sqrt_one_minus_alpha_bar'][t].view(-1, 1, 1, 1) * noise)
            loss = torch.mean((noise - model(x_t, t)) ** 2)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_history.append(loss.item())
    return model, schedule, loss_history


# --------------------------------------------------------------------------- #
# Samplers
# --------------------------------------------------------------------------- #
def make_steps(T_train, T_infer):
    idxs = torch.linspace(0, T_train - 1, T_infer, dtype=torch.long)
    return list(reversed(idxs.tolist()))


def sample_ddpm(model, schedule, n_samples, steps=None):
    model.eval()
    T = schedule['T']
    if steps is None:
        steps = list(range(T - 1, -1, -1))
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    with torch.no_grad():
        for i in steps:
            t = torch.full((n_samples,), i, device=device, dtype=torch.long)
            eps = model(x, t)
            alpha, alpha_bar, beta = schedule['alphas'][i], schedule['alpha_bar'][i], schedule['betas'][i]
            x = (1. / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1. - alpha_bar)) * eps)
            if i > 0:
                x = x + torch.sqrt(beta) * torch.randn_like(x)
    return x.clamp(-1., 1.)


def sample_ddim(model, schedule, n_samples, steps):
    model.eval()
    ab = schedule['alpha_bar']
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    with torch.no_grad():
        for i, j in zip(steps[:-1], steps[1:]):
            t = torch.full((n_samples,), i, device=device, dtype=torch.long)
            eps = model(x, t)
            x0 = (x - torch.sqrt(1. - ab[i]) * eps) / torch.sqrt(ab[i])
            x = torch.sqrt(ab[j]) * x0 + torch.sqrt(1. - ab[j]) * eps
    return x.clamp(-1., 1.)


# --------------------------------------------------------------------------- #
# FID (Inception-v3 pool features, matched to the paper)
# --------------------------------------------------------------------------- #
def get_inception():
    try:
        net = tv_models.inception_v3(weights=tv_models.Inception_V3_Weights.IMAGENET1K_V1,
                                     transform_input=False)
    except Exception:
        net = tv_models.inception_v3(pretrained=True, transform_input=False)
    net.fc = nn.Identity(); net.eval()
    return net.to(device)


def _prep_inception(imgs):
    imgs = (imgs + 1.) / 2.
    imgs = torch.nn.functional.interpolate(imgs, size=(299, 299), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=imgs.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=imgs.device).view(1, 3, 1, 1)
    return (imgs.repeat(1, 3, 1, 1) - mean) / std


def _activations(inception, source, max_samples):
    feats, seen = [], 0
    with torch.no_grad():
        if isinstance(source, DataLoader):
            for x, _ in source:
                f = inception(_prep_inception(x.to(device))); feats.append(f.cpu().numpy())
                seen += f.size(0)
                if seen >= max_samples: break
        else:  # callable(batch_size)->images in [-1,1]
            while seen < max_samples:
                cur = min(128, max_samples - seen)
                f = inception(_prep_inception(source(cur).to(device))); feats.append(f.cpu().numpy())
                seen += cur
    return np.concatenate(feats, 0)[:max_samples]


def _fid(mu1, s1, mu2, s2):
    diff = mu1 - mu2
    eig = np.real(np.linalg.eigvals(s1.dot(s2))); eig[eig < 0] = 0
    return float(diff.dot(diff) + np.trace(s1) + np.trace(s2) - 2 * np.sum(np.sqrt(eig)))


def fid_for_generator(gen_fn, train_ds, n_real=3000, n_gen=3000, inception=None):
    inception = inception or get_inception()
    rf = _activations(inception, DataLoader(train_ds, batch_size=128, shuffle=True), n_real)
    gf = _activations(inception, gen_fn, n_gen)
    return _fid(rf.mean(0), np.cov(rf, rowvar=False), gf.mean(0), np.cov(gf, rowvar=False))



### Config


In [ ]:
CHECKPOINTS  = [10, 30, 60]     # training-budget checkpoints (epochs); model trains once to max
STEP_COUNTS  = [100, 200, 400, 700, 1000]
SEEDS        = [0, 1, 2]
N_REAL = N_GEN = 3000
import json, numpy as np, torch
inception = get_inception()
print('device:', device, '| max epochs per model:', max(CHECKPOINTS))


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:01<00:00, 106MB/s] 


device: cpu | max epochs per model: 60


### Train once to max budget, evaluate FID at each checkpoint


In [ ]:
def train_and_eval_checkpoints(T, checkpoints, loader, train_ds):
    schedule = make_schedule(T)
    model = SimpleUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    cps = sorted(checkpoints); out = {}
    for ep in range(1, cps[-1] + 1):
        model.train()
        for x, _ in loader:
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_t = (schedule['sqrt_alpha_bar'][t].view(-1,1,1,1) * x +
                   schedule['sqrt_one_minus_alpha_bar'][t].view(-1,1,1,1) * noise)
            loss = torch.mean((noise - model(x_t, t)) ** 2)
            opt.zero_grad(); loss.backward(); opt.step()
        if ep in cps:
            fid = fid_for_generator(lambda bs: sample_ddpm(model, schedule, bs),
                                    train_ds, N_REAL, N_GEN, inception)
            out[ep] = fid
            print(f'    epoch {ep:>3}: FID={fid:.1f}')
    return out

runs = {ep: {T: [] for T in STEP_COUNTS} for ep in CHECKPOINTS}
for seed in SEEDS:
    set_seed(seed)
    _, loader = mnist_loader(batch_size=128, train=True)
    for T in STEP_COUNTS:
        set_seed(seed)
        print(f'seed={seed} T={T}')
        res = train_and_eval_checkpoints(T, CHECKPOINTS, loader, loader.dataset)
        for ep in CHECKPOINTS: runs[ep][T].append(res[ep])


100%|██████████| 9.91M/9.91M [00:00<00:00, 51.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 2.17MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.34MB/s]


seed=0 T=100


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### Results: FID mean +/- std for each budget, and a combined table


In [ ]:
summary = {}
for ep in CHECKPOINTS:
    print(f'\n=== Budget {ep} epochs: FID mean +/- std over seeds ===')
    summary[ep] = {}
    for T in STEP_COUNTS:
        a = np.array(runs[ep][T]); m, s = float(a.mean()), float(a.std(ddof=1) if len(a)>1 else 0.0)
        summary[ep][T] = (m, s)
        print(f'  T={T:>4}: {m:6.1f} +/- {s:4.1f}   runs={np.round(a,1).tolist()}')

print('\n=== Combined table: rows = budget (epochs), cols = T, cells = FID mean+/-std ===')
print('epochs\\T  ' + '  '.join(f'{T:>12}' for T in STEP_COUNTS))
for ep in CHECKPOINTS:
    print(f'{ep:>6}   ' + '  '.join(f'{summary[ep][T][0]:5.1f}+/-{summary[ep][T][1]:<4.1f}' for T in STEP_COUNTS))
json.dump({str(ep): {str(T): summary[ep][T] for T in STEP_COUNTS} for ep in CHECKPOINTS},
          open('budget_sweep.json','w'), indent=2)
print('\nSaved budget_sweep.json')


### How to read this
- If the FID gaps between T values stay smaller than the +/- std at every budget, the step-count null result holds across budgets (a strong statement).
- If a clear, consistent ordering of T emerges at the larger budgets (gaps exceed the std), that is a real, budget-dependent effect worth reporting; send me the combined table and I will reframe the paper and the response letter around it.
- Either way, report FID as mean +/- std and state the budget(s) used. Send me `budget_sweep.json` or the printed combined table.
